# Hotel Booking Cancellation Prediction — FINAL Random Forest Model

This notebook produces the **final, deployable model**: Random Forest, full feature set
(no feature selection), hyperparameter-tuned with an emphasis on **reducing overfitting**
rather than chasing the last 0.1% of accuracy.

### What changed from the previous tuning run
The earlier `RandomizedSearchCV` picked `bootstrap=False`, `max_depth=None`,
`min_samples_leaf=2` — a combination that let trees memorize the training set
(Train Accuracy 0.987 vs Test Accuracy 0.848, a 14-point gap = overfitting), for a
barely-there +0.15% test accuracy gain over the untuned baseline.

This version **constrains the search space** to force regularized trees:
- `bootstrap` is always `True` (each tree only sees a random subsample of rows)
- `max_depth` is capped (no unlimited-depth trees)
- `min_samples_leaf` / `min_samples_split` have a higher floor (leaves can't be built from
  just 1-2 samples, which is what causes memorization)

The result should be a model with test accuracy close to before, but a much smaller
train/test gap — i.e. a model you can actually trust to generalize.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv("/content/hotel_bookings.csv")

print("Dataset Loaded Successfully!")

df.head()


In [ ]:
print("Shape of Dataset :", df.shape)
print("\nMissing Values\n")
print(df.isnull().sum())
print("\nDuplicate Rows :", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates()

print("Duplicate Rows After :", df.duplicated().sum())


In [ ]:
# Fill missing values

if 'children' in df.columns:
    df['children'].fillna(df['children'].median(), inplace=True)

if 'country' in df.columns:
    df['country'].fillna(df['country'].mode()[0], inplace=True)

if 'agent' in df.columns:
    df['agent'].fillna(0, inplace=True)

if 'company' in df.columns:
    df['company'].fillna(0, inplace=True)

print(df.isnull().sum())


In [ ]:
# Feature Engineering

df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

df["total_stay"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

df["total_cost"] = (
    df["adr"] *
    df["total_stay"]
)

df.head()


## Remove Data Leakage

`reservation_status` and `reservation_status_date` are only known **after** the booking outcome
is decided — dropping them is mandatory before any training.


In [ ]:
leak_cols = ["reservation_status", "reservation_status_date"]

leak_cols_present = [c for c in leak_cols if c in df.columns]
print("Dropping leakage columns:", leak_cols_present)

df = df.drop(columns=leak_cols_present)

print("Remaining columns:", df.shape[1])


In [ ]:
# Label Encoding

encoder = LabelEncoder()

cat_cols = df.select_dtypes(include='object').columns

# Keep track of per-column encoders is not needed here since we fit one shared
# LabelEncoder object sequentially -- if you need to inverse_transform later,
# fit a separate LabelEncoder per column instead. Kept as-is for consistency
# with earlier notebooks.
for col in cat_cols:
    df[col] = encoder.fit_transform(df[col].astype(str))

print("Encoding Completed")


In [ ]:
# Full feature set -- no feature selection

X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

assert not any(c in X.columns for c in leak_cols), "Leakage column found in X!"

print(X.shape)
print(y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Hyperparameter Tuning — Regularized Search Space

The key difference from before: this grid **cannot** produce the memorization-prone
combination that caused overfitting last time.


In [ ]:
param_dist = {
    "n_estimators": [200, 300, 400, 500],
    "max_depth": [6, 8, 10, 12, 15],           # capped -- no unlimited depth
    "min_samples_split": [10, 15, 20, 30],      # raised floor
    "min_samples_leaf": [4, 6, 8, 10],          # raised floor -- prevents tiny memorizing leaves
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True],                        # always sample -- adds regularization
    "class_weight": [None, "balanced"]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,                  # 5-fold for a more reliable estimate
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    random_state=42,
    return_train_score=True
)

random_search.fit(X_train_scaled, y_train)

print("Best Parameters Found:")
print(random_search.best_params_)

print("\nBest Cross-Validation Accuracy :", round(random_search.best_score_, 4))


## Check the Train/Test Gap Across the Top Candidates

Rather than blindly taking whichever candidate scored highest on CV accuracy, we look at
the candidates' train vs. validation score gap too, to confirm we're not quietly picking
an overfit model again.


In [ ]:
cv_results = pd.DataFrame(random_search.cv_results_)

cv_results["gap"] = cv_results["mean_train_score"] - cv_results["mean_test_score"]

summary_cols = [
    "params",
    "mean_train_score",
    "mean_test_score",
    "gap",
    "rank_test_score"
]

top10 = cv_results.sort_values("rank_test_score").head(10)[summary_cols]

pd.set_option("display.max_colwidth", None)
top10


In [ ]:
best_rf = random_search.best_estimator_

# Predictions on the held-out test set
y_pred = best_rf.predict(X_test_scaled)
y_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

# Metrics
train_acc = best_rf.score(X_train_scaled, y_train)
test_acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
cv_score = cross_val_score(best_rf, X_train_scaled, y_train, cv=5).mean()

diff = train_acc - test_acc
if diff <= 0.05:
    status = "Good Fit"
elif diff <= 0.10:
    status = "Slight Overfit"
else:
    status = "Overfitting"

print("="*60)
print("FINAL Random Forest -- Regularized & Tuned")
print("="*60)
print("Train Accuracy :", round(train_acc, 4))
print("Test Accuracy  :", round(test_acc, 4))
print("Precision      :", round(precision, 4))
print("Recall         :", round(recall, 4))
print("F1 Score       :", round(f1, 4))
print("ROC AUC        :", round(roc, 4))
print("CV Score       :", round(cv_score, 4))
print("Train/Test Gap :", round(diff, 4))
print("Status         :", status)


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix -- FINAL Random Forest")
plt.show()


In [ ]:
print("="*60)
print("Classification Report -- FINAL Random Forest")
print("="*60)
print(classification_report(y_test, y_pred))


In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"FINAL Random Forest (AUC = {roc:.4f})")
plt.plot([0,1], [0,1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve -- FINAL Random Forest")
plt.legend()
plt.show()


In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10,6))
sns.barplot(x=importances.values, y=importances.index)
plt.title("Top 15 Feature Importances -- FINAL Random Forest")
plt.xlabel("Importance")
plt.show()

print(importances)


## Test the Final Model on 10 Random Records


In [ ]:
sample_data = df.sample(n=10, random_state=42)

X_sample = sample_data.drop("is_canceled", axis=1)
y_actual = sample_data["is_canceled"]

X_sample_scaled = scaler.transform(X_sample)

y_pred_sample = best_rf.predict(X_sample_scaled)
y_prob_sample = best_rf.predict_proba(X_sample_scaled)[:, 1]

result_df = sample_data.copy()
result_df["Actual"] = y_actual.values
result_df["Predicted"] = y_pred_sample
result_df["Correct"] = result_df["Actual"] == result_df["Predicted"]
result_df["Probability"] = y_prob_sample

label_map = {0: "Not Cancelled", 1: "Cancelled"}
result_df["Actual Label"] = result_df["Actual"].map(label_map)
result_df["Predicted Label"] = result_df["Predicted"].map(label_map)

display_columns = ["Actual", "Predicted", "Actual Label", "Predicted Label", "Probability", "Correct"]
print(result_df[display_columns])

correct = result_df["Correct"].sum()
print("\n" + "="*60)
print("Correct Predictions :", correct)
print("Wrong Predictions   :", 10 - correct)
print("Sample Accuracy     :", round((correct/10)*100, 2), "%")
print("="*60)


## Finalize the Model

Save the trained model, the fitted scaler, and a metadata file (feature order, best
hyperparameters, final metrics) so the model can be reloaded and used for prediction
later without re-running the whole notebook.


In [ ]:
import os

os.makedirs("/content/final_model", exist_ok=True)

joblib.dump(best_rf, "/content/final_model/random_forest_final.pkl")
joblib.dump(scaler, "/content/final_model/scaler.pkl")

feature_order = list(X.columns)

metadata = {
    "model": "RandomForestClassifier",
    "feature_order": feature_order,
    "best_params": random_search.best_params_,
    "metrics": {
        "train_accuracy": round(float(train_acc), 4),
        "test_accuracy": round(float(test_acc), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1_score": round(float(f1), 4),
        "roc_auc": round(float(roc), 4),
        "cv_score": round(float(cv_score), 4),
        "train_test_gap": round(float(diff), 4),
        "status": status
    }
}

import json as _json
with open("/content/final_model/metadata.json", "w") as f:
    _json.dump(metadata, f, indent=2)

print("Saved:")
print(" - /content/final_model/random_forest_final.pkl")
print(" - /content/final_model/scaler.pkl")
print(" - /content/final_model/metadata.json")


In [ ]:
# Example: how to reload and use the final model later
#
# import joblib
# import pandas as pd
#
# model = joblib.load("/content/final_model/random_forest_final.pkl")
# scaler = joblib.load("/content/final_model/scaler.pkl")
#
# new_data = pd.DataFrame([...])          # must have the same columns as feature_order
# new_data_scaled = scaler.transform(new_data[feature_order])
# prediction = model.predict(new_data_scaled)
# probability = model.predict_proba(new_data_scaled)[:, 1]

print("Reload example printed above as a comment -- copy into a fresh notebook/script when needed.")


## Final Summary


In [ ]:
print("="*60)
print("FINAL MODEL SUMMARY")
print("="*60)
print("Algorithm        : Random Forest (all features, no feature selection)")
print("Best Hyperparameters:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")
print("-"*60)
print("Test Accuracy    :", round(test_acc, 4))
print("ROC AUC          :", round(roc, 4))
print("Precision        :", round(precision, 4))
print("Recall           :", round(recall, 4))
print("F1 Score         :", round(f1, 4))
print("Train/Test Gap   :", round(diff, 4))
print("Fit Status       :", status)
print("="*60)
